In [1]:
#%pip install scipy
#%pip install statsmodels

In [1]:
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
from scipy.stats import spearmanr, pearsonr
import glob

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# === CONFIGURAZIONE ===
# percorso della cartella con i CSV puliti
#path = "/content/drive/MyDrive/Desktop/Magistrale/IoT_ESP_SleepSense/Misurazioni/*.csv"
path = "C:/Users/leopi/Il mio Drive/Desktop/Magistrale/IoT_ESP_SleepSense/Misurazioni/*.csv"
# === CARICAMENTO FILES ===
all_files = glob.glob(path)
dfs = []

for file in all_files:
    df = pd.read_csv(file)
    dfs.append(df)

# unisci tutti i file in un unico dataframe
df_all = pd.concat(dfs, ignore_index=True)

# === PULIZIA E TIPI ===
df_all["_time"] = pd.to_datetime(df_all["_time"], utc=True, errors="coerce", format="mixed")
cols_to_numeric = ["humidity", "light", "mic", "temperature", "is_moving"]
df_all[cols_to_numeric] = df_all[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# rimuovi eventuali righe con valori nulli
df_all = df_all.dropna(subset=cols_to_numeric)

print("Dimensione dataset:", df_all.shape)

Dimensione dataset: (2520, 16)


In [3]:
# === ANALISI CORRELAZIONE ===
# matrice di correlazione Pearson
corr_matrix = df_all[cols_to_numeric].corr(method="pearson")

# calcolo anche Spearman tra movimento e altri sensori
for col in ["humidity", "light", "mic", "temperature"]:
    spear_corr, _ = spearmanr(df_all["is_moving"], df_all[col])
    pear_corr, _ = pearsonr(df_all["is_moving"], df_all[col])
    print(f"\nCorrelazione con {col}:")
    print(f"  Pearson:  {pear_corr:.3f}")
    print(f"  Spearman: {spear_corr:.3f}")




Correlazione con humidity:
  Pearson:  0.228
  Spearman: 0.237

Correlazione con light:
  Pearson:  0.119
  Spearman: 0.105

Correlazione con mic:
  Pearson:  -0.086
  Spearman: -0.184

Correlazione con temperature:
  Pearson:  -0.502
  Spearman: -0.390


In [4]:
# === GRAFICI ===
# 1. Line chart di tutte le variabili
fig = px.line(df_all, x="_time", y=cols_to_numeric, title="Plot delle variabili nel tempo")
fig.update_layout(xaxis_title="Time", yaxis_title="Sensor Values")
fig.show()


# 2. Heatmap delle correlazioni
fig_corr = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    annotation_text=corr_matrix.round(2).values,
    showscale=True,
    colorscale="Viridis",
    reversescale=True
)
fig_corr.update_layout(title="Correlation Heatmap (Pearson)")
fig_corr.show()

# 3. Scatter plot movimento vs ogni variabile
for col in ["humidity", "light", "mic", "temperature"]:
    fig_scatter = px.scatter(df_all, x=col, y="is_moving", trendline="ols",
                             title=f"Movimento vs {col}")
    fig_scatter.show()
